# KV Cache en LLMs: qué es, qué reta, y qué significa para ML Research

**Autor:** Carlos Alberto León Gil — [github.com/CarlosALeon](https://github.com/CarlosALeon)
Ingeniero industrial · Maestría en Ciencia de Datos · Bogotá, Colombia

---

Este notebook parte del tutorial de **Sebastian Raschka**, *Understanding and Coding the KV Cache in LLMs from Scratch* ([enlace](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms)), y lo aterriza desde la óptica de **investigación en ML**.

La tesis: extraer el *concepto* del KV cache es la parte fácil. Lo que importa para un investigador es entender **el reto que abre** (memoria que crece sin control) y **lo que significa** (un trade-off cómputo↔memoria que hoy define la agenda de inferencia eficiente en LLMs).

## 1. El problema, en una frase

Un LLM genera texto **un token a la vez**. En cada paso reprocesa toda la secuencia anterior para predecir el siguiente token.

El detalle que lo cambia todo: en atención causal, cada token se proyecta en tres vectores —query (Q), key (K), value (V)— y **los K/V de los tokens ya vistos no dependen de los tokens futuros**. Son invariantes. Recalcularlos en cada paso es trabajo tirado a la basura.

- **Por qué son invariantes:** `k = x·W_k` y `v = x·W_v` dependen solo del token `x` y de pesos fijos. El token 1 produce el mismo `k(1), v(1)` sin importar qué venga después.
- **Por qué el query no se cachea:** solo necesitamos el query del token *actual*, porque la atención es `softmax(q·Kᵀ/√d)·V` con `q` = query nuevo y `K, V` = toda la historia.

## 2. La idea del KV cache

Guardar los K/V ya calculados y reutilizarlos. En cada paso nuevo:
1. Calcular K/V **solo** del token nuevo.
2. Concatenarlos a la caché.
3. Atender contra toda la caché.

Consecuencia teórica (esto es lo que citan todos los papers): el costo por paso pasa de **O(n²)** a **O(n)** lineal. A cambio, la memoria crece linealmente con la longitud de secuencia.

In [ ]:
import time
import torch
import torch.nn as nn

torch.manual_seed(123)
print("torch", torch.__version__)

## 3. Implementación desde cero: atención causal con y sin caché

El corazón del mecanismo es el bloque `if use_cache`. Todo lo demás es contabilidad.

In [ ]:
class CausalAttention(nn.Module):
    """Atención causal de una cabeza, con y sin KV cache."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        # Buffers de caché: arrancan vacíos.
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

    def reset_cache(self):
        # Por qué: entre dos prompts distintos hay que limpiar, o el modelo
        # atiende a keys viejos y produce salida incoherente.
        self.cache_k, self.cache_v = None, None

    def forward(self, x, use_cache=False):
        q     = self.W_query(x)
        k_new = self.W_key(x)
        v_new = self.W_value(x)

        if use_cache:
            # Solo calculamos K/V del token nuevo; el resto se recupera.
            if self.cache_k is None:
                self.cache_k, self.cache_v = k_new, v_new
            else:
                self.cache_k = torch.cat([self.cache_k, k_new], dim=1)
                self.cache_v = torch.cat([self.cache_v, v_new], dim=1)
            k, v = self.cache_k, self.cache_v
        else:
            # Sin caché: K/V se recalculan sobre TODA la secuencia cada vez.
            k, v = k_new, v_new

        # Atención causal enmascarada.
        scores = q @ k.transpose(-2, -1) / (self.d_out ** 0.5)
        Tq, Tk = q.shape[1], k.shape[1]
        offset = Tk - Tq  # desde qué posición absoluta arrancan los queries
        mask = torch.triu(
            torch.ones(Tq, Tk, device=x.device, dtype=torch.bool),
            diagonal=1 + offset,
        )
        scores = scores.masked_fill(mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        return attn @ v

### Por qué el `offset` en la máscara

Sin caché, query y keys tienen la misma longitud → máscara triangular normal.

Con caché, en *decode* pasamos **1 query** contra **Tk keys** (toda la historia). Ese query está en la posición absoluta `Tk-1`, así que debe ver *todas* las keys. `offset = Tk - Tq` corre la diagonal para alinear. Este tipo de detalle de indexación es justo donde Raschka advierte que "es fácil equivocarse y divergir".

## 4. Un mini-GPT autorregresivo

Segundo detalle sutil: el **tracking de posición** (`current_pos`). Los positional embeddings de los tokens nuevos deben continuar donde quedó la caché, no reiniciar en 0.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, ctx_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(ctx_len, d_model)
        self.att = CausalAttention(d_model, d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.ctx_len = ctx_len
        self.current_pos = 0  # cuántos tokens ya cacheamos

    def reset_cache(self):
        self.att.reset_cache()
        self.current_pos = 0

    def forward(self, idx, use_cache=False):
        b, T = idx.shape
        if use_cache:
            # Las posiciones nuevas continúan donde quedamos.
            pos = torch.arange(self.current_pos, self.current_pos + T, device=idx.device)
            self.current_pos += T
        else:
            pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos).unsqueeze(0)
        x = self.att(x, use_cache=use_cache)
        return self.head(x)

## 5. Generación: prefill + decode

- **Prefill:** procesar el prompt completo una vez (llena la caché).
- **Decode:** por cada token nuevo, alimentar **solo** ese token. Ahí está el ahorro.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, use_cache):
    model.eval()
    if use_cache:
        model.reset_cache()
        logits = model(idx, use_cache=True)          # prefill
        for _ in range(max_new_tokens):
            next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_idx], dim=1)
            logits = model(next_idx, use_cache=True)  # decode: SOLO el token nuevo
    else:
        for _ in range(max_new_tokens):
            logits = model(idx[:, -model.ctx_len:], use_cache=False)  # reprocesa todo
            next_idx = logits[:, -1].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_idx], dim=1)
    return idx

## 6. Correctitud primero, velocidad después

Regla de oro: una optimización que cambia el resultado **no** es optimización, es un bug. Antes de medir velocidad, verificamos que con y sin caché se produce **exactamente** la misma secuencia.

In [ ]:
vocab_size, d_model, ctx_len = 100, 64, 512
model = MiniGPT(vocab_size, d_model, ctx_len)
prompt = torch.randint(0, vocab_size, (1, 4))
N = 200

out_no  = generate(model, prompt.clone(), N, use_cache=False)
out_yes = generate(model, prompt.clone(), N, use_cache=True)

same = torch.equal(out_no, out_yes)
print("Salidas idénticas (correctitud):", same)
assert same, "El KV cache cambió el resultado -> hay un bug de indexación."

## 7. El speed-up

Tiempo de generar 200 tokens con y sin caché. El modelo no está entrenado (genera ruido), pero da igual: el KV cache es optimización de *inferencia*; no cambia *qué* se genera, solo *qué tan rápido*.

In [ ]:
def timeit(use_cache, reps=3):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        generate(model, prompt.clone(), N, use_cache=use_cache)
        ts.append(time.perf_counter() - t0)
    return min(ts)

t_no  = timeit(False)
t_yes = timeit(True)
print(f"Sin cache:  {t_no:.3f} s")
print(f"Con cache:  {t_yes:.3f} s")
print(f"Speed-up:   {t_no / t_yes:.2f}x")

### Cómo escala el ahorro con la longitud

La teoría dice O(n²) → O(n): a más tokens, mayor speed-up. Esto es lo que hace al KV cache indispensable en contexto largo.

In [ ]:
def run(uc, n):
    t0 = time.perf_counter()
    generate(model, prompt.clone(), n, uc)
    return time.perf_counter() - t0

print(f"{'n':>5} {'sin_cache':>10} {'con_cache':>10} {'speed-up':>9}")
for n in [50, 100, 200, 400]:
    a = min(run(False, n) for _ in range(2))
    b = min(run(True,  n) for _ in range(2))
    print(f"{n:5d} {a:10.3f} {b:10.3f} {a/b:8.2f}x")

## 8. Lo que significa para ML Research

El concepto es simple. Lo importante es lo que **abre**:

**El trade-off central.** Cambiamos cómputo (O(n²)→O(n)) por memoria (crece lineal con la secuencia). En modelos grandes y contextos largos, la caché KV puede pesar más que el propio modelo. Ese es el cuello de botella real de la inferencia moderna.

**El reto no es el concepto, es la ingeniería a escala.** Raschka lo señala con dos *pitfalls* que ya son líneas de investigación:
- El `torch.cat` repetido fragmenta memoria → motivó **PagedAttention / vLLM** (gestión tipo memoria virtual).
- El crecimiento lineal → motivó **compresión, poda (eviction) y atención compartida** (MQA/GQA/MLA).

**Por qué le importa a un científico de datos.** Decidir si sirves un LLM con contexto de 4K o 128K, cuánta GPU necesitas y cuánto cuesta por token, se reduce en gran parte a cómo gestionas esta caché.

## 9. Qué dicen los papers de arXiv (10 recientes, 2026)

Google Scholar/arXiv dan el KV cache por sentado y atacan **el reto de la memoria**. Selección de este año:

1. **Learning to Evict from Key-Value Cache** — arXiv:2602.10238. Aprende *qué* tokens descartar en vez de heurísticas.
2. **Semantic-Retrieval-Guided KV-Cache Compression** — arXiv:2606.24467. Compresión guiada por recuperación semántica para contexto largo en hardware limitado.
3. **A Shared Asymmetrically-Compressed KV Cache Pool for Multi-Agent LLM Inference (PolyKV)** — arXiv:2604.24971. Reduce la caché de 19.8 GB a 0.45 GB (−97.7%) compartiéndola entre agentes.
4. **Compressed and Composable KV Cache Reuse** — arXiv:2607.17715 (KDD '26). Reutilizar caché entre prompts con prefijo compartido.
5. **KV Cache Optimization Strategies for Scalable and Efficient LLM Inference** — arXiv:2603.20397. Survey; define el KV cache como "optimización fundacional".
6. **KV Cache Transform Coding for Compact Storage** — arXiv:2511.01815. Codificación por transformada para almacenamiento compacto.
7. **Inference-Time Hyper-Scaling with KV Cache Compression** — arXiv:2506.05345. Comprimir la caché para generar *más* tokens con el mismo presupuesto.
8. **Synthesizing Recurrence with KV Cache Compression** — arXiv:2402.09398. Recurrencia + compresión para tareas que exigen recordar muchos tokens.
9. **An Efficient KV Cache Layer for Enterprise-Scale LLM Inference** — arXiv:2510.09665. Mover la caché fuera de GPU para reusarla entre queries y motores.
10. **KV Cache Compression for Inference Efficiency in LLMs: A Review** — arXiv:2508.06297. Revisión de estrategias de compresión.

Fundacionales citados una y otra vez: **MQA** (Shazeer, 2019, arXiv:1911.02150), **GQA** (Ainslie et al., 2023, arXiv:2305.13245), **H2O / Heavy-Hitter Oracle** (arXiv:2306.14048), **PagedAttention/vLLM** (Kwon et al., 2023).

## 10. Dónde está el código (GitHub)

- **Fuente original de este notebook:** Sebastian Raschka, `LLMs-from-scratch/ch04/03_kv-cache` — [github.com/rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch04/03_kv-cache).
- **vLLM (PagedAttention en producción):** [github.com/vllm-project/vllm](https://github.com/vllm-project/vllm).
- **Este ejercicio, adaptado y comentado en español:** [github.com/CarlosALeon](https://github.com/CarlosALeon).

---

### Cierre

El KV cache se aprende en 10 minutos. Pero para ML Research el valor no está en copiar el `torch.cat` — está en ver que detrás hay un trade-off cómputo↔memoria que sostiene toda una agenda: compresión, eviction, atención compartida y gestión de memoria a nivel de sistema. Extraer el concepto es el punto de partida; entender el reto y lo que significa es el trabajo real.

*Basado en el tutorial de Sebastian Raschka. Papers vía arXiv. Adaptación y notas: Carlos A. León.*